# Boundary QA

This notebook does a quick quality-assurance pass over the demo boundary files.

It checks:
- expected geography keys
- geometry types
- join coverage against the point-capable demo service-request records
- a simple geometry preview

In [ ]:
from pathlib import Path
import sys

import nyc311
from IPython.display import display

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "examples").exists() and (candidate / "src").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples.utils import configure_matplotlib_style, data_path, save_current_figure

boundaries_gdf = nyc311.load_boundaries_geodataframe(
    data_path("community_district_boundaries.geojson")
)
records = nyc311.load_service_requests(data_path("service_requests_fixture.csv"))
joined = nyc311.spatial_join_records_to_boundaries(
    nyc311.records_to_geodataframe(records),
    boundaries_gdf,
)

In [ ]:
boundary_summary = boundaries_gdf.assign(
    geometry_type=boundaries_gdf.geometry.geom_type,
)[["geography", "geography_value", "geometry_type"]]

join_coverage = joined["boundary_geography_value"].notna().mean()
print(f"Join coverage: {join_coverage:.1%}")
display(boundary_summary)

In [ ]:
plt = configure_matplotlib_style()
axes = boundaries_gdf.boundary.plot(figsize=(8, 6), color="black")
axes.set_axis_off()
axes.set_title("Community district boundary preview")
preview_path = save_current_figure("community-district-boundary-preview.png", axes.figure)
preview_path